# MedFlow — Notebook 0 · Extração de Dados

**Sprint 2 · Ômega Urban Tech · Turma 1TSCOA**

Baixa os dados públicos do DATASUS e do IBGE e materializa os **arquivos brutos**
que o notebook 1 consome. Este é o ponto de partida numa máquina limpa.

## Por que este notebook existe

Os dados pesados **não vão para o GitHub**: são 3,9 GB de `.dbc`/`.dbf` e um parquet
de 238 MB. Quem clonar o repositório precisa conseguir reconstruir tudo a partir das
fontes públicas — é isso que este notebook garante.

| | |
|---|---|
| **Entrada** | FTP `ftp.datasus.gov.br` (SIH/RD e CNES/LT) + API de localidades do IBGE |
| **Saída** | `dados/processados/*_raw.parquet` e `dados/referencias/municipios_ibge.csv` |
| **Cache** | `dados/raw/` — `.dbc` e `.dbf` por competência |
| **Recorte** | São Paulo, competências 2022-01 a 2023-12 (24 meses) |

## Encadeamento

```
FTP DATASUS  ──► dados/raw/*.dbc  ──► dados/raw/*.dbf  ──► dados/processados/*_raw.parquet
                  (download)          (descompressão)       (leitura + concatenação)
API IBGE     ──────────────────────────────────────────► dados/referencias/municipios_ibge.csv
```

Cada etapa **verifica o cache antes de trabalhar**. Rodar o notebook uma segunda vez
não baixa nada de novo, e uma execução interrompida retoma de onde parou.

## Nota sobre o `pysus`

A arquitetura da Sprint 1 declara `pysus 2.2`. Na versão 2.7 o catálogo interno
(*ducklake*) devolve apenas **3 dos 24 arquivos RD** de SP no período — a extração via
`pysus.sih()` não reproduz a base. Por isso aqui falamos direto com o FTP do DATASUS
pela `ftplib` da biblioteca padrão, que é a fonte que o próprio `pysus` espelha.
Menos dependência e mais controle.

In [1]:
from ftplib import FTP
from pathlib import Path
import json
import urllib.request

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import datasus_dbc
from dbfread import DBF

# --- Caminhos -------------------------------------------------------------
BASE    = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_RAW = BASE / "dados" / "raw"            # cache de .dbc/.dbf
DIR_OUT = BASE / "dados" / "processados"    # parquets brutos
DIR_REF = BASE / "dados" / "referencias"    # tabelas de apoio
for d in (DIR_RAW, DIR_OUT, DIR_REF):
    d.mkdir(parents=True, exist_ok=True)

# --- Parâmetros do recorte ------------------------------------------------
UF          = "SP"
COMPETENCIAS = [(a, m) for a in (2022, 2023) for m in range(1, 13)]

FTP_HOST = "ftp.datasus.gov.br"
FTP_DIRS = {
    "RD": "/dissemin/publicos/SIHSUS/200801_/Dados",      # SIH — AIH reduzida
    "LT": "/dissemin/publicos/CNES/200508_/Dados/LT",     # CNES — leitos
}

# Se False, a gravação aborta caso o parquet de saída já exista.
SOBRESCREVER = False

def nome_arquivo(grupo, ano, mes):
    '''RD + SP + 22 + 01 -> "RDSP2201.dbc" (padrão de nomenclatura do DATASUS).'''
    return f"{grupo}{UF}{str(ano)[2:]}{mes:02d}.dbc"

print("cache   :", DIR_RAW)
print("saída   :", DIR_OUT)
print(f"recorte : {UF}, {len(COMPETENCIAS)} competências "
      f"({COMPETENCIAS[0][0]}-{COMPETENCIAS[0][1]:02d} a {COMPETENCIAS[-1][0]}-{COMPETENCIAS[-1][1]:02d})")

cache   : <projeto>/dados/raw
saída   : <projeto>/dados/processados
recorte : SP, 24 competências (2022-01 a 2023-12)


---
## 1 · Download dos `.dbc` do DATASUS

Uma conexão FTP por grupo, com a listagem do diretório lida uma única vez. Arquivos
já presentes no cache são pulados — é o que torna a execução retomável.

> O FTP do DATASUS é público e anônimo. Os diretórios são grandes (o do SIH tem mais
> de 22 mil arquivos), por isso listamos uma vez e resolvemos os 24 nomes em memória.

In [2]:
def baixar_grupo(grupo, competencias):
    '''Baixa os .dbc faltantes de um grupo. Devolve (baixados, do_cache, ausentes).'''
    alvos = {nome_arquivo(grupo, a, m): (a, m) for a, m in competencias}
    faltando = {n: c for n, c in alvos.items() if not (DIR_RAW / n).exists()}

    if not faltando:
        print(f"[{grupo}] {len(alvos)} arquivos já em cache — nada a baixar.")
        return 0, len(alvos), []

    baixados, ausentes = 0, []
    ftp = FTP(FTP_HOST, timeout=120)
    try:
        ftp.login()
        ftp.cwd(FTP_DIRS[grupo])
        # o FTP mistura maiúsculas e minúsculas; indexamos por nome normalizado
        remoto = {n.upper(): n for n in ftp.nlst()}
        for nome in sorted(faltando):
            real = remoto.get(nome.upper())
            if real is None:
                ausentes.append(nome)
                print(f"[{grupo}] AUSENTE no FTP: {nome}")
                continue
            destino = DIR_RAW / nome
            parcial = destino.with_suffix(".dbc.parcial")   # nunca deixa meio arquivo válido
            with open(parcial, "wb") as fh:
                ftp.retrbinary(f"RETR {real}", fh.write)
            parcial.rename(destino)
            baixados += 1
            print(f"[{grupo}] baixado {nome} ({destino.stat().st_size/1e6:.1f} MB)")
    finally:
        try: ftp.quit()
        except Exception: ftp.close()

    return baixados, len(alvos) - len(faltando), ausentes


resumo_download = {}
for grupo in ("RD", "LT"):
    resumo_download[grupo] = baixar_grupo(grupo, COMPETENCIAS)

for grupo, (novos, cache, ausentes) in resumo_download.items():
    print(f"\n{grupo}: {novos} baixados · {cache} do cache · {len(ausentes)} ausentes")
    assert not ausentes, f"faltam arquivos no FTP para {grupo}: {ausentes}"

[RD] 24 arquivos já em cache — nada a baixar.
[LT] 24 arquivos já em cache — nada a baixar.

RD: 0 baixados · 24 do cache · 0 ausentes

LT: 0 baixados · 24 do cache · 0 ausentes


---
## 2 · Descompressão `.dbc` → `.dbf`

`.dbc` é o dBase comprimido proprietário do DATASUS. A biblioteca `datasus-dbc`
descomprime para `.dbf` padrão, que o `dbfread` lê.

Os `.dbf` ocupam bem mais espaço que os `.dbc` (3,9 GB somados) — por isso
`dados/raw/` é ignorado pelo Git.

In [3]:
convertidos, em_cache = 0, 0
for grupo in ("RD", "LT"):
    for ano, mes in COMPETENCIAS:
        dbc = DIR_RAW / nome_arquivo(grupo, ano, mes)
        dbf = dbc.with_suffix(".dbf")
        if dbf.exists():
            em_cache += 1
            continue
        datasus_dbc.decompress(str(dbc), str(dbf))
        convertidos += 1
        print(f"convertido {dbf.name} ({dbf.stat().st_size/1e6:.1f} MB)")

print(f"\n{convertidos} convertidos · {em_cache} já existiam")
print(f"cache ocupa {sum(f.stat().st_size for f in DIR_RAW.glob('*')) / 1e9:.2f} GB")


0 convertidos · 48 já existiam
cache ocupa 4.09 GB


---
## 3 · Contrato de tipos

O DBF descreve tipos por campo, mas competências diferentes ocasionalmente divergem
(uma coluna sai como inteiro num mês e como texto noutro). Se isso passar batido, o
parquet final ganha colunas de tipo misto e o notebook 1 quebra.

Declaramos o contrato explicitamente: **estas colunas são numéricas, todo o resto é
texto.** Códigos como `CNES`, `MUNIC_MOV` e `DIAG_PRINC` são texto de propósito —
tratá-los como número comeria o zero à esquerda.

In [4]:
SIH_INT = [
    "UTI_MES_IN", "UTI_MES_AN", "UTI_MES_AL", "UTI_MES_TO",
    "UTI_INT_IN", "UTI_INT_AN", "UTI_INT_AL", "UTI_INT_TO",
    "DIAR_ACOM", "QT_DIARIAS", "RUBRICA", "IDADE", "DIAS_PERM",
    "MORTE", "TOT_PT_SP", "NUM_FILHOS", "SEQUENCIA",
]
SIH_FLOAT = [
    "VAL_SH", "VAL_SP", "VAL_SADT", "VAL_RN", "VAL_ACOMP", "VAL_ORTP",
    "VAL_SANGUE", "VAL_SADTSR", "VAL_TRANSP", "VAL_OBSANG", "VAL_PED1AC",
    "VAL_TOT", "VAL_UTI", "US_TOT",
    "VAL_SH_FED", "VAL_SP_FED", "VAL_SH_GES", "VAL_SP_GES", "VAL_UCI",
]
LT_INT = ["QT_EXIST", "QT_CONTR", "QT_SUS", "QT_NSUS"]

CONTRATO = {
    "RD": {"int": SIH_INT,  "float": SIH_FLOAT},
    "LT": {"int": LT_INT,   "float": []},
}

def aplicar_contrato(df, grupo):
    '''Coage as colunas ao contrato de tipos; o restante vira texto sem espaços.'''
    regras = CONTRATO[grupo]
    for c in regras["int"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("int64")
    for c in regras["float"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype("float64")
    numericas = set(regras["int"]) | set(regras["float"])
    for c in df.columns:
        if c not in numericas:
            df[c] = df[c].astype("string").str.strip()
    return df

print("colunas numéricas declaradas — RD:", len(SIH_INT) + len(SIH_FLOAT), "| LT:", len(LT_INT))

colunas numéricas declaradas — RD: 36 | LT: 4


---
## 4 · Leitura e consolidação

Vinte e quatro `.dbf` por grupo viram **um** parquet. A gravação é **incremental**:
cada competência é escrita e liberada da memória antes da próxima. Concatenar os
5,2 milhões de linhas do SIH de uma vez custaria vários GB de RAM — assim o notebook
roda em máquina modesta.

Também derivamos aqui `_ano` e `_mes`, as colunas de competência que todo o resto do
projeto usa como chave temporal:

- **SIH** — de `ANO_CMPT` / `MES_CMPT`;
- **CNES** — de `COMPETEN` (formato `AAAAMM`).

In [5]:
def competencia_do_df(df, grupo, ano, mes):
    '''Deriva _ano e _mes a partir dos campos do próprio arquivo, com fallback no nome.'''
    if grupo == "RD" and {"ANO_CMPT", "MES_CMPT"} <= set(df.columns):
        df["_ano"] = pd.to_numeric(df.ANO_CMPT, errors="coerce").fillna(ano).astype("int64")
        df["_mes"] = pd.to_numeric(df.MES_CMPT, errors="coerce").fillna(mes).astype("int64")
    elif grupo == "LT" and "COMPETEN" in df.columns:
        comp = df.COMPETEN.astype("string").str.strip()
        df["_ano"] = pd.to_numeric(comp.str[:4], errors="coerce").fillna(ano).astype("int64")
        df["_mes"] = pd.to_numeric(comp.str[4:6], errors="coerce").fillna(mes).astype("int64")
    else:
        df["_ano"], df["_mes"] = ano, mes
    return df


def consolidar(grupo, arquivo_saida):
    '''Lê os 24 .dbf do grupo e grava um único parquet, competência a competência.'''
    destino = DIR_OUT / arquivo_saida
    if destino.exists() and not SOBRESCREVER:
        print(f"[{grupo}] {arquivo_saida} já existe — pulando "
              f"(defina SOBRESCREVER = True para refazer).")
        return None

    temp = destino.with_suffix(".parquet.parcial")
    escritor, total, esquema_ref = None, 0, None
    try:
        for ano, mes in COMPETENCIAS:
            dbf = (DIR_RAW / nome_arquivo(grupo, ano, mes)).with_suffix(".dbf")
            df = pd.DataFrame(iter(DBF(str(dbf), encoding="iso-8859-1")))
            df = aplicar_contrato(df, grupo)
            df = competencia_do_df(df, grupo, ano, mes)

            if esquema_ref is None:
                esquema_ref = list(df.columns)
            else:
                # alinha ao esquema da primeira competência: colunas novas somem,
                # colunas ausentes entram nulas — o parquet fica retangular
                df = df.reindex(columns=esquema_ref)

            tabela = pa.Table.from_pandas(df, preserve_index=False)
            if escritor is None:
                escritor = pq.ParquetWriter(temp, tabela.schema, compression="snappy")
            escritor.write_table(tabela.cast(escritor.schema))

            total += len(df)
            print(f"[{grupo}] {ano}-{mes:02d}  {len(df):>8,} linhas  (acumulado {total:>9,})")
            del df, tabela
    finally:
        if escritor is not None:
            escritor.close()

    temp.rename(destino)
    print(f"\n[{grupo}] {destino.name}: {total:,} linhas · {destino.stat().st_size/1e6:.1f} MB")
    return total

total_sih  = consolidar("RD", "sih_sp_2022_2023_raw.parquet")
total_cnes = consolidar("LT", "cnes_lt_sp_2022_2023_raw.parquet")

[RD] sih_sp_2022_2023_raw.parquet já existe — pulando (defina SOBRESCREVER = True para refazer).
[LT] cnes_lt_sp_2022_2023_raw.parquet já existe — pulando (defina SOBRESCREVER = True para refazer).


---
## 5 · Tabela de municípios do IBGE

O SIH identifica município por código de **6 dígitos**; o IBGE usa **7**. Sem esta
tabela o painel exibe `350570` no lugar de `Barueri` e nenhum cruzamento com dados
externos (população, PIB, malha geográfica) é possível.

Fonte: API pública de localidades do IBGE, estado 35 (São Paulo).

In [6]:
URL_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/35/municipios"
ARQ_IBGE = DIR_REF / "municipios_ibge.csv"

if ARQ_IBGE.exists() and not SOBRESCREVER:
    municipios = pd.read_csv(ARQ_IBGE, dtype=str)
    print(f"{ARQ_IBGE.name} já existe — {len(municipios)} municípios lidos do cache.")
else:
    with urllib.request.urlopen(URL_IBGE, timeout=60) as resposta:
        bruto = json.loads(resposta.read().decode("utf-8"))
    municipios = pd.DataFrame([{
        "codigo_ibge7": str(m["id"]),
        "nome":         m["nome"],
        "microrregiao": m["microrregiao"]["nome"],
        "mesorregiao":  m["microrregiao"]["mesorregiao"]["nome"],
    } for m in bruto])
    municipios.to_csv(ARQ_IBGE, index=False)
    print(f"{len(municipios)} municípios gravados em {ARQ_IBGE.name}")

# O código de 6 dígitos do SIH é o de 7 sem o dígito verificador
municipios["codigo_ibge6"] = municipios.codigo_ibge7.str[:6]
print(municipios.head(5).to_string(index=False))

municipios_ibge.csv já existe — 645 municípios lidos do cache.
codigo_ibge7             nome          microrregiao           mesorregiao codigo_ibge6
     3500105       Adamantina            Adamantina   Presidente Prudente       350010
     3500204           Adolfo São José do Rio Preto São José do Rio Preto       350020
     3500303            Aguaí          Pirassununga              Campinas       350030
     3500402   Águas da Prata São João da Boa Vista              Campinas       350040
     3500501 Águas de Lindóia                Amparo              Campinas       350050


---
## 6 · Validação

Confere que a extração reproduz as dimensões declaradas na Sprint 1. Se algum número
divergir, alguma competência veio incompleta e **não se deve seguir para o notebook 1**.

In [7]:
sih_chk  = pq.ParquetFile(DIR_OUT / "sih_sp_2022_2023_raw.parquet")
cnes_chk = pq.ParquetFile(DIR_OUT / "cnes_lt_sp_2022_2023_raw.parquet")

checks = [
    ("linhas SIH/RD",          sih_chk.metadata.num_rows,          5210357),
    ("colunas SIH/RD",         len(sih_chk.schema_arrow.names),    115),
    ("linhas CNES/LT",         cnes_chk.metadata.num_rows,         200075),
    ("colunas CNES/LT",        len(cnes_chk.schema_arrow.names),   30),
    ("municípios IBGE em SP",  len(municipios),                    645),
]

print(f"{'verificação':<26} {'obtido':>10} {'esperado':>10}")
print("-" * 52)
ok = True
for nome, obtido, esperado in checks:
    passou = obtido == esperado
    ok &= passou
    print(f"{nome:<26} {obtido:>10,} {esperado:>10,}   {'OK' if passou else 'DIVERGE'}")

# As colunas que o notebook 1 exige precisam existir e estar tipadas
exigidas = {"CNES", "MUNIC_MOV", "ESPEC", "QT_DIARIAS", "MORTE", "VAL_TOT",
            "DIAG_PRINC", "_ano", "_mes"}
faltando = exigidas - set(sih_chk.schema_arrow.names)
print(f"\ncolunas exigidas ausentes no SIH: {faltando or 'nenhuma'}")
ok &= not faltando

print("\n" + ("Extração válida — pode seguir para o notebook 1."
              if ok else "ATENÇÃO: divergência na extração. Não seguir."))

verificação                    obtido   esperado
----------------------------------------------------
linhas SIH/RD               5,210,357  5,210,357   OK
colunas SIH/RD                    115        115   OK
linhas CNES/LT                200,075    200,075   OK
colunas CNES/LT                    30         30   OK
municípios IBGE em SP             645        645   OK

colunas exigidas ausentes no SIH: nenhuma

Extração válida — pode seguir para o notebook 1.


---
## O que fica pronto

| Arquivo | Conteúdo |
|---|---|
| `dados/processados/sih_sp_2022_2023_raw.parquet` | 5.210.357 AIH × 115 colunas |
| `dados/processados/cnes_lt_sp_2022_2023_raw.parquet` | 200.075 registros de leito × 30 colunas |
| `dados/referencias/municipios_ibge.csv` | 645 municípios de SP, código de 7 dígitos e nome |
| `dados/raw/` | cache de 48 `.dbc` + 48 `.dbf` (~3,9 GB, fora do Git) |

**Próximo passo:** `01_engenharia_dados.ipynb`, que transforma estes brutos nas bases
curadas.

### Numa máquina limpa

```bash
uv venv .venv && uv pip install --python .venv/bin/python \
    pandas pyarrow matplotlib seaborn jupyter datasus-dbc dbfread
```

Depois execute os notebooks em ordem: `00` → `01` → `02`. A primeira execução do `00`
baixa cerca de 400 MB de `.dbc` e expande para ~3,9 GB em `.dbf`; as seguintes usam o
cache. Nada em `dados/` vai para o repositório.